# RL Group Project: Starter Notebook
## Clinical Treatment Optimisation: Sepsis ICU Management

**Master in Data Science & Advanced Analytics — Reinforcement Learning Course**

This project is structured in two stages of increasing complexity.

- In **Configuration A**, you will work with a tabular Sepsis MDP, where the state and action spaces are small enough to apply classical RL methods directly.

- In **Configuration B**, you will move to a continuous-observation ICU environment that is clinically grounded and significantly more challenging.

Three realistic failure modes are present in Configuration B, each reflecting a real scenario encountered in clinical AI deployments. The first is episodic observation noise, where monitoring equipment occasionally malfunctions. The second is episodic missing observations, representing situations where lab results are simply unavailable for an entire episode. The third is acute clinical events, which are sudden and irreversible patient deteriorations that occur independently of any treatment given.

---
### Group Members
Group V

```
Student 1: Alano Gonçalves 20250457
Student 2: Catarina Martins 20221914
Student 3: João Carichas 20250507
Student 4: Marta Ribeiro 20221886
Student 5: Nicole Nogueira 20221961
```


---
## Setup Instructions

### Requirements
Run the following installation cell before running the notebook for the first time:

```bash
pip install icu-sepsis numpy pandas matplotlib seaborn tqdm optuna stable-baselines3 torch
```

### Directory Structure
The following files must be present in the project directory:
```
project/
├── rl_sepsis_v4.ipynb     ← this notebook
├── envs/
│   ├── env_setup.py       ← environment factory and constants
│   └── wrappers.py        ← clinical reality wrappers
```

### Running the Notebook
Run all cells **top to bottom** in order (Kernel → Restart & Run All).

- Configuration A trains tabular RL agents (~30-40 minutes)
- Configuration B trains deep RL agents (~40-60 minutes)
- Creative extension runs reward shaping experiments (~20 minutes)
- Total estimated runtime: **90-120 minutes**

### Reproducibility
All experiments use `SEED = 42`. All evaluation functions use an isolated
`RandomState(seed)` so results are fully reproducible across runs.
Evaluation uses **5,000 episodes** throughout to minimise stochastic variance.


---
## 0. Setup & Imports


In [ ]:
# Install dependencies (run once)
#!pip install icu-sepsis numpy pandas matplotlib seaborn tqdm optuna stable-baselines3 torch

import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', message='.*Gym.*')
warnings.filterwarnings('ignore', message='.*gymnasium.*')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import os
from tqdm import tqdm
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

os.makedirs('plots', exist_ok=True)
PLOTS_DIR = 'plots'

SEED = 42
np.random.seed(SEED)

from envs.env_setup import (
    ENV_ID, N_STATES, N_ACTIONS, STATE_SURVIVED, STATE_DIED,
    GAMMA, INTENSITY, SOFA_BIAS, LAM,
    make_sepsis_env,
)

print(f'ICU-Sepsis-v2 | States: {N_STATES} | Actions: {N_ACTIONS}')
print(f'Terminal states: {STATE_SURVIVED} (survived, r=+1)  {STATE_DIED} (died, r=0)')
print('Setup complete!')


In [ ]:
# Configuration already loaded from env_setup.py
# env_setup.py defines: SOFA_BIAS=5.0, LAM=0.02, INTENSITY, make_sepsis_env()
print(f'Required config: sofa_bias={SOFA_BIAS}, lam={LAM}')


- *sofa_bias = 5.0* means patients start with an average SOFA score of about 9 — moderately sick, which is consistent with an ICU sepsis population.
- *lam = 0.02* is the small intensity penalty discouraging over-treatment — every step the agent gives intense treatment, it receives a -0.02 penalty on top of the survival reward.


---
## 1. Explore the Environment

`ICU-Sepsis-v2` is a benchmark MDP constructed from real MIMIC-III patient data. Each episode represents the trajectory of one ICU patient. The agent observes a discrete integer state (ranging from 0 to 715) and must select one of 25 treatment actions corresponding to combinations of vasopressor and IV fluid dose levels. The reward signal is sparse: **+1 at survival, 0 at death, and 0 for all intermediate steps**, with a discount factor γ = 1.


In [ ]:
#  Instantiate and inspect the raw environment
env = make_sepsis_env()
obs, info = env.reset(seed=SEED)

print(f'Observation space : {env.observation_space} discrete integer state')
print(f'Action space      : {env.action_space}')
print(f'Initial state     : {obs}')
print()

#  Extract the full MDP model
raw    = env.unwrapped
P      = raw._tx_mat                  # (716, 25, 716) — P[s,a,s'] = P(s'|s,a)
R_sasp = raw._r_mat                   # (716, 25, 716) — R[s, a, s']
R      = (P * R_sasp).sum(axis=2)    # (716, 25)      — E[r | s, a]

print(f'Transition matrix P : {P.shape}  (S x A x S\')')
print(f'Reward matrix R     : {R.shape}  (S x A)')
print(f'Reward range        : [{R.min():.3f}, {R.max():.3f}]')


By opening the "virtual hospital" and analysing it, we can understand that:
- The patient can be in one of **716 states** — a single discrete integer representing clinical condition
- At each step, the agent chooses from **25 possible treatment actions** (5 vasopressor × 5 IV fluid levels)
- The **transition matrix P** (716 × 25 × 716) is the full rulebook — only available in Config A
- The **reward range [-0.020, 0.708]** confirms sparsity: most rewards are near 0, only terminal transitions carry meaningful signal


In [ ]:
#  Random baseline: establish the performance floor
def run_random_baseline(n_episodes=5000, seed=SEED):
    rng = np.random.RandomState(seed)
    env_eval = make_sepsis_env()
    returns, lengths = [], []
    for _ in range(n_episodes):
        obs, _ = env_eval.reset(seed=int(rng.randint(100_000)))
        total_r, steps, done = 0.0, 0, False
        while not done:
            obs, r, te, tr, _ = env_eval.step(env_eval.action_space.sample())
            total_r += r; steps += 1; done = te or tr
        returns.append(total_r)
        lengths.append(steps)
    env_eval.close()
    return np.array(returns), np.array(lengths)


rand_returns, rand_lengths = run_random_baseline()
survival_rate = float(np.mean(rand_returns > 0)) * 100

print(f'Random agent ({len(rand_returns)} episodes):')
print(f'  Mean return    : {np.mean(rand_returns):.4f}')
print(f'  Survival rate  : {survival_rate:.1f}%')
print(f'  Mean ep length : {np.mean(rand_lengths):.1f} steps')
print()
print('All Config A algorithms must beat the random baseline.')


A completely random agent saves **~70% of patients**, which is already a solid survival rate. This reflects the real MIMIC-III data — sepsis patients in ICUs have decent baseline survival regardless of treatment. However, the goal is to meaningfully improve on this baseline through intelligent treatment decisions. All algorithms must beat this floor.

> **Note on reproducibility:** All evaluation functions use `n_episodes=5000` with an isolated `RandomState(seed)` to ensure results are stable and reproducible across runs. This reduces stochastic variance from ±2pp to approximately ±0.5pp.


In [ ]:
#  Visualise state visitation and reward structure
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

rng_vis = np.random.RandomState(SEED)
env_vis = make_sepsis_env()
visited = []
for _ in range(300):
    obs, _ = env_vis.reset(seed=int(rng_vis.randint(100_000)))
    done = False
    while not done:
        visited.append(int(obs))
        obs, _, te, tr, _ = env_vis.step(env_vis.action_space.sample())
        done = te or tr
env_vis.close()
clinical = [s for s in visited if s not in (STATE_SURVIVED, STATE_DIED)]

axes[0].hist(clinical, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_xlabel('State index (0-713 = clinical)')
axes[0].set_ylabel('Visit count')
axes[0].set_title('State visitation random policy (300 episodes)', fontweight='bold')

axes[1].bar(range(N_STATES), R.max(axis=1), color='tomato', width=1.0, alpha=0.8)
axes[1].set_xlabel('State index')
axes[1].set_ylabel('max_a R(s,a)')
axes[1].set_title('Reward sparsity — only terminal transitions carry reward', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_env_exploration.png', bbox_inches='tight')
plt.show()


---
# Config A — Tabular Methods

With 716 discrete states and 25 actions, the Q-table has shape `(716, 25)`, totalling 17,900 entries. This size is entirely manageable in memory, which is precisely what motivates the use of tabular algorithms here.

We implement three algorithms in Config A:
- **Policy Iteration (PI)**: a model-based Dynamic Programming method that uses the full MDP (transition matrix P and reward matrix R) to find the theoretically optimal policy. Serves as the performance ceiling.
- **SARSA**: an on-policy, model-free TD algorithm. Learns from environment interaction; its updates use the action the agent actually takes next, making it conservative.
- **Q-Learning**: an off-policy, model-free TD algorithm. Also learns from interaction but always updates towards the best possible next action, making it more aggressive.

The natural contrast: **PI** has access to the full model (unfair advantage but gives us the optimal ceiling), while **SARSA and Q-Learning** must discover the policy through trial and error. Within the model-free pair, the **on-policy vs off-policy** distinction maps directly onto clinical conservatism.


## 2. Policy Iteration: Optimal Benchmark

Policy Iteration is a Dynamic Programming method that alternates between two steps:
1. **Policy Evaluation**: given the current policy, compute V(s) for all states using the Bellman expectation equation iteratively.
2. **Policy Improvement**: for each state, update the policy to the action that maximises the expected value.

Because the full MDP (P and R) is available in Config A, PI finds the exact optimal policy without any interaction with the environment. This makes it the **theoretical ceiling** against which we measure our model-free agents.

> **Note on PI as the ceiling:** PI optimises over the full transition matrix P, which is estimated from finite MIMIC-III data. The remaining gap between the best model-free agent and PI is the irreducible cost of not having the model — not something more training can fully close. Additionally, since P is estimated from finite data, PI is not an absolute clinical upper bound but the best achievable within this tabular model.


In [ ]:
## Policy Iteration

def policy_iteration(P, R, gamma=1.0, theta=1e-8, max_iter=1000):
    """
    Exact Policy Iteration using the full MDP matrices.

    Parameters
    ----------
    P        : np.ndarray (S, A, S') — transition probabilities
    R        : np.ndarray (S, A)     — expected reward per state-action
    gamma    : float                 — discount factor
    theta    : float                 — convergence threshold
    max_iter : int                   — max improvement iterations

    Returns
    -------
    policy        : np.ndarray (S,) — optimal action per state
    V             : np.ndarray (S,) — optimal value function
    n_iter        : int             — iterations to convergence
    delta_history : list            — max delta per evaluation sweep
    """
    n_states, n_actions, _ = P.shape
    policy = np.zeros(n_states, dtype=int)
    V = np.zeros(n_states)
    delta_history = []

    for iteration in range(max_iter):
        # Step 1: Policy Evaluation
        while True:
            delta = 0.0
            for s in range(n_states):
                a = policy[s]
                v_new = R[s, a] + gamma * np.dot(P[s, a], V)
                delta = max(delta, abs(v_new - V[s]))
                V[s] = v_new
            delta_history.append(delta)
            if delta < theta:
                break

        # Step 2: Policy Improvement
        policy_stable = True
        for s in range(n_states):
            old_action = policy[s]
            Q_s = R[s] + gamma * P[s].dot(V)
            policy[s] = np.argmax(Q_s)
            if old_action != policy[s]:
                policy_stable = False

        if policy_stable:
            print(f'Policy Iteration converged after {iteration + 1} improvement iterations.')
            break

    return policy, V, iteration + 1, delta_history


np.random.seed(SEED)
pi_policy, pi_V, pi_iters, pi_deltas = policy_iteration(P, R, gamma=GAMMA)

print(f'Value function range: [{pi_V.min():.4f}, {pi_V.max():.4f}]')
print(f'Unique actions used by PI policy: {len(np.unique(pi_policy))}/25')


It converged in just 4 rounds, which is extremely fast because it is solving equations directly using the full MDP model rather than learning from experience. The value function ranges from 0.00 (states where survival is essentially impossible) to 0.98 (states near guaranteed survival). Only 15 of the 25 possible actions are ever optimal — 10 actions are never the right choice in any state.


In [ ]:
## Evaluate Policy Iteration

def evaluate_policy_tabular(policy_array, n_episodes=5000, seed=SEED):
    """
    Evaluate a deterministic tabular policy over n_episodes.
    Uses an isolated RandomState for full reproducibility across runs.
    Returns dict with mean_return, survival_rate, mean_length.
    """
    rng = np.random.RandomState(seed)
    env_eval = make_sepsis_env()
    returns, lengths = [], []

    for _ in range(n_episodes):
        obs, _ = env_eval.reset(seed=int(rng.randint(100_000)))
        total_r, steps, done = 0.0, 0, False
        while not done:
            action = int(policy_array[int(obs)])
            obs, r, te, tr, _ = env_eval.step(action)
            total_r += r; steps += 1; done = te or tr
        returns.append(total_r)
        lengths.append(steps)

    env_eval.close()
    returns = np.array(returns)
    return {
        'mean_return'   : float(np.mean(returns)),
        'survival_rate' : float(np.mean(returns > 0)) * 100,
        'mean_length'   : float(np.mean(lengths)),
    }


pi_results = evaluate_policy_tabular(pi_policy)
print('Policy Iteration — Evaluation (5000 episodes):')
print(f'  Mean return   : {pi_results["mean_return"]:.4f}')
print(f'  Survival rate : {pi_results["survival_rate"]:.1f}%')
print(f'  Mean length   : {pi_results["mean_length"]:.1f} steps')
print(f'  vs Random baseline survival: {survival_rate:.1f}%')


PI saves around **78-80% of patients** — our performance ceiling. The improvement over random (+8-10pp) shows what is possible when you have perfect knowledge of the environment. This ceiling is used as a reference throughout Config A.


In [ ]:
## Plot PI: convergence + value function + policy heatmap

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].semilogy(pi_deltas, color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Evaluation sweep')
axes[0].set_ylabel('Max |V change| (log scale)')
axes[0].set_title('Policy Iteration — Convergence', fontweight='bold')

clinical_states = [s for s in range(N_STATES) if s not in (STATE_SURVIVED, STATE_DIED)]
axes[1].bar(clinical_states, pi_V[clinical_states], color='tomato', width=1.0, alpha=0.8)
axes[1].set_xlabel('State index')
axes[1].set_ylabel('V*(s)')
axes[1].set_title('Optimal Value Function V*', fontweight='bold')

policy_display = pi_policy[:714]
vaso_levels  = policy_display // 5
fluid_levels = policy_display  % 5
axes[2].scatter(range(len(policy_display)), vaso_levels,
                c=fluid_levels, cmap='RdYlGn', s=4, alpha=0.6)
axes[2].set_xlabel('State index')
axes[2].set_ylabel('Vasopressor level (0=none, 4=high)')
axes[2].set_title('PI Policy: vasopressor level per state\n(colour = IV fluid level)', fontweight='bold')
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=plt.Normalize(0, 4))
plt.colorbar(sm, ax=axes[2], label='IV fluid level')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_PI.png', bbox_inches='tight')
plt.show()


- **Convergence Plot**: Shows the maximum change in the value function per sweep, on a log scale. It drops sharply to near zero confirming PI converged cleanly in 4 iterations.
- **Value Function Plot**: Each bar is a patient state. Tall bars = high chance of survival. Clear variation confirms the model has differentiated well between patient prognoses.
- **Policy Heatmap**: For each clinical state, shows the vasopressor level PI recommends with IV fluid level as colour. Reveals that PI uses a structured, state-dependent treatment strategy.


---
## 3. SARSA: On-Policy TD Control

SARSA (**S**tate–**A**ction–**R**eward–next **S**tate–next **A**ction) is a model-free, on-policy TD control algorithm. At each step it uses the sequence `(s, a, r, s', a')` to update the Q-value:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl[r + \gamma\, Q(s', a') - Q(s,a)\bigr]$$

The key: **a' is selected by the same epsilon-greedy policy** the agent is currently following. This means SARSA's updates account for the risk of future exploratory (random) actions — making it naturally more conservative than Q-Learning. In a clinical context this conservatism is desirable: the agent effectively learns a policy that is safe even while it is still uncertain.


In [ ]:
## SARSA Training

def sarsa(n_episodes=50_000, alpha=0.3, gamma=1.0,
          epsilon_start=1.0, epsilon_min=0.01, seed=SEED):
    """
    SARSA on-policy TD control for the tabular Sepsis MDP.

    Parameters
    ----------
    n_episodes    : int   — number of training episodes
    alpha         : float — learning rate
    gamma         : float — discount factor
    epsilon_start : float — initial exploration probability
    epsilon_min   : float — minimum exploration probability
    seed          : int   — random seed

    Returns
    -------
    Q           : np.ndarray (N_STATES, N_ACTIONS) — learned Q-table
    returns_log : list of float — episode returns during training
    """
    np.random.seed(seed)
    env_train = make_sepsis_env()
    Q = np.zeros((N_STATES, N_ACTIONS))
    returns_log = []
    epsilon = epsilon_start
    decay = (epsilon_start - epsilon_min) / n_episodes

    for ep in tqdm(range(n_episodes), desc='SARSA', leave=False):
        obs, _ = env_train.reset(seed=np.random.randint(100_000))
        s = int(obs)
        if np.random.random() < epsilon:
            a = env_train.action_space.sample()
        else:
            a = int(np.argmax(Q[s]))

        total_r, done = 0.0, False
        while not done:
            obs_next, r, te, tr, _ = env_train.step(a)
            s_next = int(obs_next)
            done = te or tr
            if np.random.random() < epsilon:
                a_next = env_train.action_space.sample()
            else:
                a_next = int(np.argmax(Q[s_next]))
            td_target = r + gamma * Q[s_next, a_next] * (not done)
            Q[s, a] += alpha * (td_target - Q[s, a])
            s, a = s_next, a_next
            total_r += r

        returns_log.append(total_r)
        epsilon = max(epsilon_min, epsilon - decay)

    env_train.close()
    return Q, returns_log


# Run SARSA with default hyperparameters
sarsa_Q, sarsa_returns = sarsa(n_episodes=50_000, alpha=0.3, gamma=GAMMA)
sarsa_policy = np.argmax(sarsa_Q, axis=1)
print('SARSA training complete.')
print(f'Unique actions used: {len(np.unique(sarsa_policy))}/25')


Unlike PI, SARSA explores all 25 actions because it has to try everything to learn what works. With default settings (α=0.3, 50k episodes) SARSA has not yet converged — the optimisation in Section 5 will address this.


In [ ]:
## SARSA Evaluation
sarsa_results = evaluate_policy_tabular(sarsa_policy)
print('SARSA — Evaluation (5000 episodes):')
print(f'  Mean return   : {sarsa_results["mean_return"]:.4f}')
print(f'  Survival rate : {sarsa_results["survival_rate"]:.1f}%')
print(f'  Mean length   : {sarsa_results["mean_length"]:.1f} steps')


At default settings, SARSA matches or slightly exceeds the random baseline. It learned something but not much yet — 50k episodes with α=0.3 is insufficient for full convergence. The optimisation section will find the right learning rate and extend training.


In [ ]:
## Plot SARSA: learning curve + Q-value distribution

window = 500
sarsa_rolling = pd.Series(sarsa_returns).rolling(window).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(sarsa_returns, alpha=0.15, color='steelblue', linewidth=0.5, label='Episode return')
axes[0].plot(sarsa_rolling, color='steelblue', linewidth=2.0, label=f'Rolling mean ({window})')
axes[0].axhline(pi_results['mean_return'], color='tomato', linestyle='--', linewidth=1.5, label='PI ceiling')
axes[0].axhline(np.mean(rand_returns), color='gray', linestyle=':', linewidth=1.5, label='Random baseline')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Return')
axes[0].set_title('SARSA — Learning Curve', fontweight='bold')
axes[0].legend(fontsize=8)

max_Q_sarsa = sarsa_Q[:714].max(axis=1)
axes[1].hist(max_Q_sarsa, bins=40, color='steelblue', edgecolor='none', alpha=0.8)
axes[1].set_xlabel('max_a Q(s,a)'); axes[1].set_ylabel('State count')
axes[1].set_title('SARSA — Q-value Distribution (clinical states)', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_SARSA.png', bbox_inches='tight')
plt.show()


- **Learning Curve Plot**: Episode returns over training with a 500-episode rolling average. PI ceiling and random baseline shown as reference lines. The curve is trying to climb but has not yet reached convergence — confirming more training and better hyperparameters are needed.
- **Q-value Distribution Plot**: Shows the spread of maximum Q-values across clinical states. The differentiation between states confirms the agent has learned that some situations are better than others, even at this early stage.


---
## 4. Q-Learning: Off-Policy TD Control

Q-Learning is a model-free, **off-policy** TD control algorithm. Its update rule differs from SARSA in one crucial place — instead of using the action the agent actually takes next (`a'`), it always bootstraps from the **maximum** Q-value in the next state:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \bigl[r + \gamma \max_{a'} Q(s', a') - Q(s,a)\bigr]$$

This means Q-Learning learns the value of the **greedy policy** regardless of how the agent actually behaves during training. It tends to converge to a more aggressive optimal policy but may overestimate Q-values due to maximisation bias. In a clinical context, Q-Learning may recommend more aggressive treatments than SARSA.


In [ ]:
## Q-Learning Training

def q_learning(n_episodes=50_000, alpha=0.3, gamma=1.0,
               epsilon_start=1.0, epsilon_min=0.01, seed=SEED):
    """
    Q-Learning off-policy TD control for the tabular Sepsis MDP.

    Parameters
    ----------
    n_episodes    : int   — number of training episodes
    alpha         : float — learning rate
    gamma         : float — discount factor
    epsilon_start : float — initial exploration probability
    epsilon_min   : float — minimum exploration probability
    seed          : int   — random seed

    Returns
    -------
    Q           : np.ndarray (N_STATES, N_ACTIONS) — learned Q-table
    returns_log : list of float — episode returns during training
    """
    np.random.seed(seed)
    env_train = make_sepsis_env()
    Q = np.zeros((N_STATES, N_ACTIONS))
    returns_log = []
    epsilon = epsilon_start
    decay = (epsilon_start - epsilon_min) / n_episodes

    for ep in tqdm(range(n_episodes), desc='Q-Learning', leave=False):
        obs, _ = env_train.reset(seed=np.random.randint(100_000))
        s = int(obs)
        total_r, done = 0.0, False

        while not done:
            if np.random.random() < epsilon:
                a = env_train.action_space.sample()
            else:
                a = int(np.argmax(Q[s]))
            obs_next, r, te, tr, _ = env_train.step(a)
            s_next = int(obs_next)
            done = te or tr
            td_target = r + gamma * np.max(Q[s_next]) * (not done)
            Q[s, a] += alpha * (td_target - Q[s, a])
            s = s_next
            total_r += r

        returns_log.append(total_r)
        epsilon = max(epsilon_min, epsilon - decay)

    env_train.close()
    return Q, returns_log


# Run Q-Learning with default hyperparameters
ql_Q, ql_returns = q_learning(n_episodes=50_000, alpha=0.3, gamma=GAMMA)
ql_policy = np.argmax(ql_Q, axis=1)
print('Q-Learning training complete.')
print(f'Unique actions used: {len(np.unique(ql_policy))}/25')


In [ ]:
## Q-Learning Evaluation
ql_results = evaluate_policy_tabular(ql_policy)
print('Q-Learning — Evaluation (5000 episodes):')
print(f'  Mean return   : {ql_results["mean_return"]:.4f}')
print(f'  Survival rate : {ql_results["survival_rate"]:.1f}%')
print(f'  Mean length   : {ql_results["mean_length"]:.1f} steps')


Q-Learning already shows improvement over the random baseline at default settings. Its off-policy updates allow it to learn the optimal policy more directly even while still exploring, giving it an early advantage over SARSA at the same default hyperparameters.


In [ ]:
## Plot Q-Learning: learning curve + SARSA vs QL overlay

window = 500
ql_rolling         = pd.Series(ql_returns).rolling(window).mean()
sarsa_rolling_full = pd.Series(sarsa_returns).rolling(window).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(ql_returns, alpha=0.15, color='darkorange', linewidth=0.5, label='Episode return')
axes[0].plot(ql_rolling, color='darkorange', linewidth=2.0, label=f'Rolling mean ({window})')
axes[0].axhline(pi_results['mean_return'], color='tomato', linestyle='--', linewidth=1.5, label='PI ceiling')
axes[0].axhline(np.mean(rand_returns), color='gray', linestyle=':', linewidth=1.5, label='Random baseline')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Return')
axes[0].set_title('Q-Learning — Learning Curve', fontweight='bold')
axes[0].legend(fontsize=8)

axes[1].plot(sarsa_rolling_full, color='steelblue', linewidth=2.0, label='SARSA')
axes[1].plot(ql_rolling, color='darkorange', linewidth=2.0, label='Q-Learning')
axes[1].axhline(pi_results['mean_return'], color='tomato', linestyle='--', linewidth=1.5, label='PI ceiling')
axes[1].axhline(np.mean(rand_returns), color='gray', linestyle=':', linewidth=1.5, label='Random baseline')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Rolling mean return')
axes[1].set_title('SARSA vs Q-Learning — Head to Head', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_QL.png', bbox_inches='tight')
plt.show()


- **Q-Learning Learning Curve Plot**: Same format as SARSA. Shows Q-Learning climbing faster toward the PI ceiling due to its off-policy updates.
- **SARSA vs Q-Learning Head-to-Head**: Both rolling means on the same axes. At default settings Q-Learning visibly outperforms SARSA, but neither has converged — the optimisation section will find the best configuration for each.


## 5. Hyperparameter Optimisation

The model-free agents in Sections 3 and 4 used default hyperparameters (alpha=0.3) and only 50,000 episodes. Looking at the learning curves, Q-Learning showed modest improvement while SARSA matched the random baseline — neither algorithm had fully converged with these settings.

Therefore, we have decided to run a focused optimisation with three goals:
1. **Bayesian alpha search** using Optuna for both SARSA and Q-Learning — smarter than a manual grid search, finding the best learning rate in fewer trials
2. **Definitive model selection** — compare Optuna best models against 500k extended training, and select the genuinely best Q-table for each algorithm using 5,000-episode evaluation
3. **Stability analysis** — in a clinical setting, a consistent policy matters as much as a high average survival rate

> **Note on PI as the ceiling:** PI optimises over the full transition matrix P, which is estimated from finite data. The remaining gap between the best model-free agent and PI is the irreducible cost of not having the model, which is not something more training can fully close.

> **Why Optuna instead of a manual grid search:** A manual grid search over 5 alpha values × 2 algorithms × 200k episodes would test only predetermined values. Optuna uses Bayesian optimisation (TPE sampler) to intelligently focus search on promising alpha regions, achieving better results in fewer trials.


In [ ]:
## Optuna hyperparameter search: SARSA and Q-Learning
# Bayesian optimisation to find the best alpha for each algorithm.
# Each trial trains for 200k episodes and evaluates with 5,000 episodes.
# 10 trials per algorithm — sufficient for Optuna to converge on a 1D search.

N_TRIALS       = 10
N_OPT_EPISODES = 200_000

# Store best Q-tables for reuse in definitive comparison and creative extension
optuna_best_Qtables  = {}   # keyed by algo_name
optuna_best_returns  = {}   # training returns of best trial per algo
optuna_all_results   = {}   # (algo, trial_number) -> {alpha, survival_rate}

def make_objective(algo_name, algo_fn):
    """
    Returns an Optuna objective function for the given algorithm.
    Samples alpha in log-scale [0.01, 0.5], trains, evaluates with 5k episodes.
    """
    def objective(trial):
        alpha = trial.suggest_float('alpha', 0.01, 0.5, log=True)
        Q, returns_tmp = algo_fn(
            n_episodes=N_OPT_EPISODES,
            alpha=alpha,
            gamma=GAMMA,
            epsilon_start=1.0,
            epsilon_min=0.01,
            seed=SEED,
        )
        policy  = np.argmax(Q, axis=1)
        results = evaluate_policy_tabular(policy, seed=SEED)
        sr      = results['survival_rate']

        # Store Q-table if best so far
        if (algo_name not in optuna_best_Qtables or
                sr > optuna_best_Qtables[algo_name]['survival_rate']):
            optuna_best_Qtables[algo_name] = {
                'Q'            : Q,
                'survival_rate': sr,
                'alpha'        : alpha,
                'returns'      : returns_tmp,
            }

        optuna_all_results[(algo_name, trial.number)] = {
            'alpha'        : alpha,
            'survival_rate': sr,
        }
        return sr
    return objective


# ── SARSA study ───────────────────────────────────────────────────────────────
print('── Optuna study: SARSA ──')
sarsa_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
sarsa_study.optimize(
    make_objective('SARSA', sarsa),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)
best_alpha_sarsa   = sarsa_study.best_params['alpha']
best_sarsa_optuna  = sarsa_study.best_value
print(f'  Best alpha : {best_alpha_sarsa:.4f}')
print(f'  Best survival : {best_sarsa_optuna:.1f}%')
print()

# ── Q-Learning study ──────────────────────────────────────────────────────────
print('── Optuna study: Q-Learning ──')
ql_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
ql_study.optimize(
    make_objective('Q-Learning', q_learning),
    n_trials=N_TRIALS,
    show_progress_bar=True,
)
best_alpha_ql  = ql_study.best_params['alpha']
best_ql_optuna = ql_study.best_value
print(f'  Best alpha : {best_alpha_ql:.4f}')
print(f'  Best survival : {best_ql_optuna:.1f}%')
print()

print('Optuna search complete.')
print(f'  SARSA best      : α={best_alpha_sarsa:.4f} → {best_sarsa_optuna:.1f}%')
print(f'  Q-Learning best : α={best_alpha_ql:.4f} → {best_ql_optuna:.1f}%')


The Optuna search explored the alpha space intelligently using Bayesian optimisation. Unlike a fixed grid, it focused trials on promising regions after initial exploration. Both algorithms' best alpha values are now stored for use in the definitive comparison.


In [ ]:
## Plot: Optuna optimisation history

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, study, algo, color in [
    (axes[0], sarsa_study, 'SARSA',      'steelblue'),
    (axes[1], ql_study,    'Q-Learning', 'darkorange'),
]:
    trials    = study.trials
    alphas    = [t.params['alpha'] for t in trials]
    survivals = [t.value           for t in trials]
    best_a    = study.best_params['alpha']

    sc = ax.scatter(alphas, survivals, c=range(len(trials)),
                    cmap='viridis', s=100, zorder=3, label='Trial')
    ax.axvline(best_a, color=color, linestyle='--', linewidth=2,
               label=f'Best α={best_a:.4f}')
    ax.axhline(pi_results['survival_rate'], color='tomato', linestyle='--',
               linewidth=1.5, label=f'PI ceiling ({pi_results["survival_rate"]:.1f}%)')
    ax.axhline(survival_rate, color='gray', linestyle=':',
               linewidth=1.5, label=f'Random ({survival_rate:.1f}%)')
    ax.set_xlabel('Alpha value')
    ax.set_ylabel('Survival Rate (%)')
    ax.set_title(f'{algo}: Optuna Search Results\n({N_TRIALS} trials, {N_OPT_EPISODES//1000}k episodes each)',
                 fontweight='bold')
    ax.set_ylim(60, 85)
    ax.legend(fontsize=8)
    plt.colorbar(sc, ax=ax, label='Trial order (dark=later)')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_optuna_search.png', bbox_inches='tight')
plt.show()


Each dot represents one Optuna trial. The colour shows trial order — darker dots are later trials, showing how Optuna progressively focused on the most promising alpha region. The vertical dashed line marks the best alpha found. This visualisation confirms that Optuna's Bayesian search explored efficiently rather than randomly.


In [ ]:
## Definitive model selection
# We compare two candidate models per algorithm:
#   1. Optuna best (200k episodes, best alpha from search)
#   2. Extended training (500k episodes, same best alpha)
# The winner is selected by 5,000-episode evaluation with fixed seed.
# We do NOT assume 500k is always better — we let the evaluation decide.

print('Training extended models (500k episodes, best alpha per algorithm)...')
print()

print(f'Q-Learning | alpha={best_alpha_ql:.4f} | 500k episodes...')
ql_opt_Q, ql_opt_returns = q_learning(
    n_episodes=500_000, alpha=best_alpha_ql,
    gamma=GAMMA, epsilon_start=1.0, epsilon_min=0.01, seed=SEED,
)
ql_opt_policy  = np.argmax(ql_opt_Q, axis=1)
ql_opt_results = evaluate_policy_tabular(ql_opt_policy, seed=SEED)
print(f'  500k survival: {ql_opt_results["survival_rate"]:.1f}%')

print()
print(f'SARSA | alpha={best_alpha_sarsa:.4f} | 500k episodes...')
sarsa_opt_Q, sarsa_opt_returns = sarsa(
    n_episodes=500_000, alpha=best_alpha_sarsa,
    gamma=GAMMA, epsilon_start=1.0, epsilon_min=0.01, seed=SEED,
)
sarsa_opt_policy  = np.argmax(sarsa_opt_Q, axis=1)
sarsa_opt_results = evaluate_policy_tabular(sarsa_opt_policy, seed=SEED)
print(f'  500k survival: {sarsa_opt_results["survival_rate"]:.1f}%')

print()
print('── Definitive comparison ──')
print(f'SARSA  Optuna (200k): {best_sarsa_optuna:.1f}%  |  500k: {sarsa_opt_results["survival_rate"]:.1f}%')
print(f'Q-Lrn  Optuna (200k): {best_ql_optuna:.1f}%  |  500k: {ql_opt_results["survival_rate"]:.1f}%')

# Select best policy per algorithm — let evaluation decide, not assumptions
sarsa_final_policy  = sarsa_opt_policy  if sarsa_opt_results['survival_rate'] >= best_sarsa_optuna else np.argmax(optuna_best_Qtables['SARSA']['Q'], axis=1)
sarsa_final_returns = sarsa_opt_returns if sarsa_opt_results['survival_rate'] >= best_sarsa_optuna else optuna_best_Qtables['SARSA']['returns']
sarsa_final_label   = f'500k (α={best_alpha_sarsa:.4f})' if sarsa_opt_results['survival_rate'] >= best_sarsa_optuna else f'Optuna 200k (α={best_alpha_sarsa:.4f})'

ql_final_policy  = ql_opt_policy  if ql_opt_results['survival_rate'] >= best_ql_optuna else np.argmax(optuna_best_Qtables['Q-Learning']['Q'], axis=1)
ql_final_returns = ql_opt_returns if ql_opt_results['survival_rate'] >= best_ql_optuna else optuna_best_Qtables['Q-Learning']['returns']
ql_final_label   = f'500k (α={best_alpha_ql:.4f})' if ql_opt_results['survival_rate'] >= best_ql_optuna else f'Optuna 200k (α={best_alpha_ql:.4f})'

print()
print(f'Best SARSA model      : {sarsa_final_label}')
print(f'Best Q-Learning model : {ql_final_label}')


Rather than assuming that more training always produces a better policy, we let the 5,000-episode evaluation decide which model — Optuna 200k or extended 500k — is genuinely better for each algorithm. This avoids the inconsistency of reporting higher numbers from one evaluation context while using worse models in the final analysis.


In [ ]:
## Stability analysis: return distribution + mean ± std
# A reliable policy has low std — in medicine, consistency is as important
# as average performance. We evaluate over 5,000 episodes for stable estimates.

def evaluate_with_distribution(policy_array, n_episodes=5000, seed=SEED):
    """
    Evaluate policy and return full return distribution + stats.
    Uses isolated RandomState for full reproducibility.
    """
    rng = np.random.RandomState(seed)
    env_eval = make_sepsis_env()
    returns = []
    for _ in range(n_episodes):
        obs, _ = env_eval.reset(seed=int(rng.randint(100_000)))
        total_r, done = 0.0, False
        while not done:
            action = int(policy_array[int(obs)])
            obs, r, te, tr, _ = env_eval.step(action)
            total_r += r; done = te or tr
        returns.append(total_r)
    env_eval.close()
    returns = np.array(returns)
    return {
        'mean_return'   : float(np.mean(returns)),
        'std_return'    : float(np.std(returns)),
        'survival_rate' : float(np.mean(returns > 0)) * 100,
        'returns_array' : returns,
    }


# Evaluate best final policies
sarsa_best_dist = evaluate_with_distribution(sarsa_final_policy)
ql_best_dist    = evaluate_with_distribution(ql_final_policy)
pi_dist         = evaluate_with_distribution(pi_policy)
rand_dist       = {
    'mean_return'   : float(np.mean(rand_returns)),
    'std_return'    : float(np.std(rand_returns)),
    'survival_rate' : survival_rate,
    'returns_array' : rand_returns,
}

print(f'PI ceiling   — survival: {pi_dist["survival_rate"]:.1f}%  std: {pi_dist["std_return"]:.4f}')
print(f'SARSA        — survival: {sarsa_best_dist["survival_rate"]:.1f}%  std: {sarsa_best_dist["std_return"]:.4f}  ({sarsa_final_label})')
print(f'Q-Learning   — survival: {ql_best_dist["survival_rate"]:.1f}%  std: {ql_best_dist["std_return"]:.4f}  ({ql_final_label})')
print(f'Random       — survival: {rand_dist["survival_rate"]:.1f}%  std: {rand_dist["std_return"]:.4f}')


Here, we evaluate the best final policy for each algorithm over 5,000 episodes to get a reliable estimate of both performance and consistency. A lower standard deviation means a more predictable and reliable policy — in a clinical context this matters as much as average survival rate, since inconsistent treatment decisions are clinically undesirable.


In [ ]:
## Plot: stability comparison

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dist_configs = [
    ('Random Baseline',                          rand_dist,       'gray'),
    ('Policy Iteration',                         pi_dist,         'tomato'),
    (f'SARSA ({sarsa_final_label})',             sarsa_best_dist, 'steelblue'),
    (f'Q-Learning ({ql_final_label})',           ql_best_dist,    'darkorange'),
]

# Return distributions
for name, d, color in dist_configs:
    axes[0].hist(d['returns_array'], bins=30, alpha=0.45,
                 color=color, label=name, density=True, edgecolor='none')
axes[0].set_xlabel('Episode Return')
axes[0].set_ylabel('Density')
axes[0].set_title('Return Distributions (5000 episodes)', fontweight='bold')
axes[0].legend(fontsize=8)

# Mean ± std
names  = [d[0].replace(' (', '\n(') for d in dist_configs]
means  = [d[1]['mean_return'] for d in dist_configs]
stds   = [d[1]['std_return']  for d in dist_configs]
colors = [d[2]                for d in dist_configs]
axes[1].bar(range(len(names)), means, color=colors, alpha=0.8, edgecolor='white')
axes[1].errorbar(range(len(names)), means, yerr=stds,
                 fmt='none', color='black', capsize=5, linewidth=1.5)
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels(names, fontsize=8)
axes[1].set_ylabel('Mean Return ± Std')
axes[1].set_title('Mean Return with Variability\n(lower error bar = more reliable)',
                  fontweight='bold')
for i, (m, s) in enumerate(zip(means, stds)):
    axes[1].text(i, m + s + 0.005, f'σ={s:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_stability.png', bbox_inches='tight')
plt.show()


- **Return Distributions Plot**: Overlaid histograms showing the spread of outcomes for each algorithm. Narrower distribution = more reliable policy.
- **Mean ± Std Plot**: The height shows the average return and the error bars show variability. PI has the narrowest bars (most consistent), random has the widest (least consistent). SARSA and Q-Learning sit in between — both significantly more reliable than random.


In [ ]:
## Original vs Optimised: learning curves
# Show faded original (50k) and solid best model for both algorithms

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

window = 2000
sarsa_best_roll  = pd.Series(sarsa_final_returns).rolling(window).mean()
ql_best_roll     = pd.Series(ql_final_returns).rolling(window).mean()
sarsa_orig_roll  = pd.Series(sarsa_returns).rolling(500).mean()
ql_orig_roll     = pd.Series(ql_returns).rolling(500).mean()

# Left: SARSA
axes[0].plot(sarsa_orig_roll, color='steelblue', linewidth=1.0, alpha=0.35,
             label='SARSA original (50k, α=0.3)')
axes[0].plot(sarsa_best_roll, color='steelblue', linewidth=2.2,
             label=f'SARSA best ({sarsa_final_label})')
axes[0].axhline(pi_results['mean_return'], color='tomato', linestyle='--',
                linewidth=1.5, label=f'PI ceiling ({pi_results["survival_rate"]:.1f}%)')
axes[0].axhline(np.mean(rand_returns), color='gray', linestyle=':',
                linewidth=1.5, label=f'Random ({survival_rate:.1f}%)')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Rolling mean return')
axes[0].set_title('SARSA: Original vs Best', fontweight='bold')
axes[0].legend(fontsize=8)

# Right: Q-Learning
axes[1].plot(ql_orig_roll, color='darkorange', linewidth=1.0, alpha=0.35,
             label='Q-Learning original (50k, α=0.3)')
axes[1].plot(ql_best_roll, color='darkorange', linewidth=2.2,
             label=f'Q-Learning best ({ql_final_label})')
axes[1].axhline(pi_results['mean_return'], color='tomato', linestyle='--',
                linewidth=1.5, label=f'PI ceiling ({pi_results["survival_rate"]:.1f}%)')
axes[1].axhline(np.mean(rand_returns), color='gray', linestyle=':',
                linewidth=1.5, label=f'Random ({survival_rate:.1f}%)')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Rolling mean return')
axes[1].set_title('Q-Learning: Original vs Best', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_optimised_curves.png', bbox_inches='tight')
plt.show()


These plots show the learning curves of both algorithms before and after optimisation. The faded thin lines represent the original 50k default runs, while the solid thick lines correspond to the best model found through Optuna search and definitive comparison. The improvement is visually clear — the solid lines sit higher than the faded ones, confirming that proper hyperparameter tuning genuinely helped.


## 6. Config A — Final Comparison & Analysis

We now bring together all algorithms tested in Configuration A — the random baseline, Policy Iteration, and the original and best versions of SARSA and Q-Learning — and compare them across three dimensions: survival rate, mean return, and policy stability.

The comparison focuses on three key metrics:
- **Survival rate**: the primary clinical metric — what percentage of patients survived under each policy
- **Mean return**: the average episodic reward, incorporating both the survival signal and the treatment intensity penalty (λ = 0.02)
- **Policy stability (std)**: the standard deviation of returns across evaluation episodes — lower value means a more consistent and reliable policy, which in a clinical setting is as important as raw performance

Key questions this section answers:
1. How much did Optuna optimisation improve each algorithm?
2. Does SARSA or Q-Learning get closer to the PI ceiling after optimisation?
3. Which model-free algorithm is more reliable in practice?
4. What do the policies actually prescribe — and does it align with clinical evidence?


In [ ]:
## Final results table
# All values sourced from the same 5,000-episode evaluation with fixed seed
# for full internal consistency

final_table = pd.DataFrame([
    {'Algorithm': 'Random Baseline', 'Version': '—',
     'Episodes': '—', 'Alpha': '—',
     'Survival %': round(survival_rate, 1),
     'Mean Return': round(float(np.mean(rand_returns)), 4),
     'Std Return': round(float(np.std(rand_returns)), 4),
     'Gap to PI (pp)': round(pi_results['survival_rate'] - survival_rate, 1)},

    {'Algorithm': 'Policy Iteration', 'Version': 'Optimal ceiling',
     'Episodes': '—', 'Alpha': '—',
     'Survival %': round(pi_dist['survival_rate'], 1),
     'Mean Return': round(pi_dist['mean_return'], 4),
     'Std Return': round(pi_dist['std_return'], 4),
     'Gap to PI (pp)': 0.0},

    {'Algorithm': 'SARSA', 'Version': 'Original',
     'Episodes': '50k', 'Alpha': '0.3',
     'Survival %': round(sarsa_results['survival_rate'], 1),
     'Mean Return': round(sarsa_results['mean_return'], 4),
     'Std Return': round(float(np.std(sarsa_returns[-5000:])), 4),
     'Gap to PI (pp)': round(pi_dist['survival_rate'] - sarsa_results['survival_rate'], 1)},

    {'Algorithm': 'SARSA', 'Version': f'Best ({sarsa_final_label})',
     'Episodes': sarsa_final_label.split(' ')[0], 'Alpha': f'{best_alpha_sarsa:.4f}',
     'Survival %': round(sarsa_best_dist['survival_rate'], 1),
     'Mean Return': round(sarsa_best_dist['mean_return'], 4),
     'Std Return': round(sarsa_best_dist['std_return'], 4),
     'Gap to PI (pp)': round(pi_dist['survival_rate'] - sarsa_best_dist['survival_rate'], 1)},

    {'Algorithm': 'Q-Learning', 'Version': 'Original',
     'Episodes': '50k', 'Alpha': '0.3',
     'Survival %': round(ql_results['survival_rate'], 1),
     'Mean Return': round(ql_results['mean_return'], 4),
     'Std Return': round(float(np.std(ql_returns[-5000:])), 4),
     'Gap to PI (pp)': round(pi_dist['survival_rate'] - ql_results['survival_rate'], 1)},

    {'Algorithm': 'Q-Learning', 'Version': f'Best ({ql_final_label})',
     'Episodes': ql_final_label.split(' ')[0], 'Alpha': f'{best_alpha_ql:.4f}',
     'Survival %': round(ql_best_dist['survival_rate'], 1),
     'Mean Return': round(ql_best_dist['mean_return'], 4),
     'Std Return': round(ql_best_dist['std_return'], 4),
     'Gap to PI (pp)': round(pi_dist['survival_rate'] - ql_best_dist['survival_rate'], 1)},
])

display(final_table)
final_table.to_csv(f'{PLOTS_DIR}/configA_final_table.csv', index=False)


From the table above, it is possible to understand the importance of hyperparameter optimisation. All values in this table come from the same 5,000-episode evaluation with fixed seed, ensuring internal consistency — there is no mixing of different evaluation contexts. The best model per algorithm was selected through definitive comparison rather than assuming more training episodes automatically produces better results.


In [ ]:
## Bar chart: survival rates + policy agreement

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bar_labels = ['Random', 'PI', 'SARSA\nOrig', f'SARSA\nBest',
              'Q-Lrn\nOrig', f'Q-Lrn\nBest']
bar_values = [survival_rate,
              pi_dist['survival_rate'],
              sarsa_results['survival_rate'],
              sarsa_best_dist['survival_rate'],
              ql_results['survival_rate'],
              ql_best_dist['survival_rate']]
bar_colors = ['gray', 'tomato', '#aed6f1', 'steelblue', '#fad7a0', 'darkorange']

bars = axes[0].bar(range(len(bar_labels)), bar_values,
                   color=bar_colors, alpha=0.9, edgecolor='white')
axes[0].set_xticks(range(len(bar_labels)))
axes[0].set_xticklabels(bar_labels, fontsize=8)
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('Config A — Survival Rate Summary', fontweight='bold')
axes[0].set_ylim(60, 85)
for bar, val in zip(bars, bar_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Policy agreement (best versions)
policy_matrix = np.stack([
    (pi_policy == sarsa_final_policy).astype(int),
    (pi_policy == ql_final_policy).astype(int),
    (sarsa_final_policy == ql_final_policy).astype(int),
], axis=0)
agreement_rates = policy_matrix.mean(axis=1) * 100
pairs = ['PI vs SARSA (best)', 'PI vs QL (best)', 'SARSA vs QL (best)']

pair_bars = axes[1].barh(pairs, agreement_rates, color='teal', alpha=0.75)
axes[1].set_xlabel('Policy Agreement (%)')
axes[1].set_title('Policy Agreement Between Algorithms\n(best versions)', fontweight='bold')
axes[1].set_xlim(0, 110)
for bar, val in zip(pair_bars, agreement_rates):
    axes[1].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_final_comparison.png', bbox_inches='tight')
plt.show()


- **Survival Rate Bar Chart**: One bar per algorithm/version. Shows the clear progression from random → original → best → PI ceiling, confirming that proper tuning meaningfully improved both algorithms.
- **Policy Agreement Plot**: Reveals how often each pair of algorithms choose the same action in the same state. The remarkably low agreement values confirm that all three algorithms converged to fundamentally different treatment strategies despite achieving similar survival rates — evidence that the sepsis MDP has multiple near-optimal policies (policy degeneracy). Clinically, this means survival rate alone is insufficient to compare policies.


In [ ]:
## Treatment Intensity Analysis: clinical interpretability
# Compare vasopressor and IV fluid doses chosen by each policy (best versions)

clinical_states_idx = [s for s in range(N_STATES)
                       if s not in (STATE_SURVIVED, STATE_DIED)]

def get_dose_distributions(policy_arr, state_idx):
    actions = policy_arr[state_idx]
    return actions // 5, actions % 5   # vaso, fluid

fig, axes = plt.subplots(2, 3, figsize=(18, 7), sharey='row')
policies_to_plot = [
    ('Policy Iteration',           pi_policy,          'tomato'),
    (f'SARSA (best)',              sarsa_final_policy,  'steelblue'),
    (f'Q-Learning (best)',         ql_final_policy,     'darkorange'),
]

for col, (name, pol, col_color) in enumerate(policies_to_plot):
    vaso, fluid = get_dose_distributions(pol, clinical_states_idx)
    axes[0, col].hist(vaso, bins=np.arange(-0.5, 5.5), color=col_color, alpha=0.8, edgecolor='white')
    axes[0, col].set_title(name, fontweight='bold', fontsize=9)
    axes[0, col].set_xlabel('Vasopressor level')
    axes[1, col].hist(fluid, bins=np.arange(-0.5, 5.5), color=col_color, alpha=0.8, edgecolor='white')
    axes[1, col].set_xlabel('IV Fluid level')

axes[0, 0].set_ylabel('State count\n(Vasopressor)', fontsize=9)
axes[1, 0].set_ylabel('State count\n(IV Fluid)', fontsize=9)

plt.suptitle('Treatment Intensity Distribution per Policy (clinical states only)',
             fontweight='bold', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_treatment_intensity.png', bbox_inches='tight')
plt.show()

print('Mean treatment intensity per algorithm (clinical states):')
print(f'{"Algorithm":<30} {"Mean Vaso":>12} {"Mean Fluid":>12}')
print('-' * 56)
for name, pol, _ in policies_to_plot:
    vaso, fluid = get_dose_distributions(pol, clinical_states_idx)
    print(f'{name:<30} {vaso.mean():>12.3f} {fluid.mean():>12.3f}')


These plots show how much vasopressor and IV fluid each algorithm recommends across all clinical states. PI recommends almost no IV fluid (≈0.07 out of 4 maximum) while model-free agents prescribe roughly 1.5 times as much. This aligns with real ICU evidence — aggressive fluid resuscitation in sepsis can worsen outcomes by causing pulmonary oedema and dilutional coagulopathy. The model-free agents likely over-learned the association between high fluid doses and survival in certain states, resulting in systematically higher prescriptions — a form of spurious correlation induced by the sparse reward signal.


In [ ]:
## Final Config A summary
print('=' * 72)
print('CONFIG A — FINAL SUMMARY')
print('=' * 72)
print(f'{"Algorithm":<25} {"Version":<20} {"Survival %":>12} {"Std":>8} {"Gap to PI":>10}')
print('-' * 72)

rows = [
    ('Random Baseline',   '—',
     survival_rate, float(np.std(rand_returns)),
     pi_dist['survival_rate'] - survival_rate),

    ('Policy Iteration',  'Ceiling',
     pi_dist['survival_rate'], pi_dist['std_return'], 0.0),

    ('SARSA',             'Original 50k',
     sarsa_results['survival_rate'], float(np.std(sarsa_returns[-5000:])),
     pi_dist['survival_rate'] - sarsa_results['survival_rate']),

    ('SARSA',             f'Best ({sarsa_final_label})',
     sarsa_best_dist['survival_rate'], sarsa_best_dist['std_return'],
     pi_dist['survival_rate'] - sarsa_best_dist['survival_rate']),

    ('Q-Learning',        'Original 50k',
     ql_results['survival_rate'], float(np.std(ql_returns[-5000:])),
     pi_dist['survival_rate'] - ql_results['survival_rate']),

    ('Q-Learning',        f'Best ({ql_final_label})',
     ql_best_dist['survival_rate'], ql_best_dist['std_return'],
     pi_dist['survival_rate'] - ql_best_dist['survival_rate']),
]

for name, version, sr, std, gap in rows:
    print(f'{name:<25} {version:<20} {sr:>11.1f}% {std:>8.4f} {gap:>9.1f}pp')

print('=' * 72)
print()

sarsa_gain = sarsa_best_dist['survival_rate'] - sarsa_results['survival_rate']
ql_gain    = ql_best_dist['survival_rate']    - ql_results['survival_rate']
print(f'Improvement from optimisation: SARSA +{sarsa_gain:.1f}pp  |  Q-Learning +{ql_gain:.1f}pp')
print()

sarsa_wins = sarsa_best_dist['survival_rate'] >= ql_best_dist['survival_rate']
sarsa_more_stable = sarsa_best_dist['std_return'] <= ql_best_dist['std_return']

if sarsa_wins:
    print(f'Best model-free algorithm: SARSA ({sarsa_best_dist["survival_rate"]:.1f}%)')
else:
    print(f'Best model-free algorithm: Q-Learning ({ql_best_dist["survival_rate"]:.1f}%)')

if sarsa_more_stable:
    print(f'Most stable: SARSA (σ={sarsa_best_dist["std_return"]:.4f}) vs Q-Learning (σ={ql_best_dist["std_return"]:.4f})')
    print('In a clinical context SARSA may be preferred: more conservative, more predictable.')
else:
    print(f'Most stable: Q-Learning (σ={ql_best_dist["std_return"]:.4f}) vs SARSA (σ={sarsa_best_dist["std_return"]:.4f})')

print()
print('Remaining gap to PI ceiling:')
print(f'  SARSA:      {pi_dist["survival_rate"] - sarsa_best_dist["survival_rate"]:.1f}pp')
print(f'  Q-Learning: {pi_dist["survival_rate"] - ql_best_dist["survival_rate"]:.1f}pp')
print()

if sarsa_best_dist['survival_rate'] > pi_dist['survival_rate']:
    print('Note: SARSA best slightly exceeds the PI evaluation benchmark.')
    print('This reflects stochasticity in evaluation episodes and noise in')
    print('the estimated transition matrix P — confirming PI is not an')
    print('absolute upper bound on clinical performance.')
    print()

print('This remaining gap reflects the cost of model-free learning.')
print('PI has access to the full MDP; model-free agents do not.')
print('Additionally, P is estimated from finite data — so PI optimises')
print('over a noisy model and is not an absolute clinical upper bound.')
